# 03 · Rules engine

Each rule is a pure function `(RuleContext) -> list[Finding]`. Because `now` is
injected (not read from the clock), results are deterministic. We evaluate
against a fixed date so the numbers match the tests.

In [ ]:
import sys, json
from pathlib import Path

# Repo root = parent of the notebooks/ directory.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
FIXTURES = REPO / "tests" / "fixtures"
print("repo:", REPO)
print("fixtures:", [p.name for p in FIXTURES.glob("*.json")])

In [ ]:
from m365_review.core.models import Organization, SubscribedSku, User, TenantData

org = Organization.from_graph(json.loads((FIXTURES / "organization.json").read_text())["value"][0])
skus = [SubscribedSku.from_graph(s) for s in json.loads((FIXTURES / "subscribedSkus.json").read_text())["value"]]
users = [User.from_graph(u) for u in json.loads((FIXTURES / "users.json").read_text())["value"]]

data = TenantData(
    organization=org, tenant_id=org.id, skus=skus, users=users,
    sign_in_activity_available=True,
)
print(f"{org.display_name}: {len(skus)} SKUs, {len(users)} users")

In [ ]:
from datetime import datetime, timezone
from m365_review.core.pricing import load_pricing
from m365_review.core.rules import run_all
from m365_review.core.rules.base import RuleContext

pricing = load_pricing(settings=None) if False else load_pricing()
NOW = datetime(2026, 7, 16, tzinfo=timezone.utc)
ctx = RuleContext(data=data, pricing=pricing, now=NOW, experimental_enabled=False)

findings = run_all(ctx)
print(f"{len(findings)} findings\n")
for f in findings:
    print(f"[{f.severity.value:6}] {f.rule_id}  {f.title}")
    print(f"         ${f.estimated_monthly_savings_usd:,.2f}/mo  ({f.affected_count} user(s))")
    for u in f.affected_users:
        print(f"           - {u.user_principal_name}: {u.detail}")
print(f"\nTOTAL: ${sum(f.estimated_monthly_savings_usd for f in findings):,.2f}/mo")

## Try the degraded path

When a tenant lacks Azure AD P1, sign-in activity is unavailable and R2 emits an
INFO finding for manual review instead of guessing:

In [ ]:
degraded = data.model_copy(update={"sign_in_activity_available": False})
ctx2 = RuleContext(data=degraded, pricing=pricing, now=NOW)
for f in run_all(ctx2):
    if f.rule_id == "R2":
        print(f.severity.value, "-", f.title)
        print(f.notes)